# Issue 5: Análisis de nulos y decisión de imputación

**Objetivo:** decidir el tratamiento de cada columna con nulos, comparar las alternativas en validación y construir un pipeline reproducible sin fuga de información.

Se continúa desde Issue 4. Las medianas y modas se ajustan solo con entrenamiento; validación se usa para comparar; prueba no participa en decisiones. `issue_d` se usa solo para analizar cohortes, nunca como predictor.

In [ ]:
%run issue4.ipynb

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_rows', 100)

## 1. Recuperar decisiones del Issue 4

Se respetan las categorías que dejó `tabla`: imputar, imputar + indicador y evaluar (más de 40 % de nulos).

In [ ]:
columnas_imputar = tabla.index[tabla['decisión'].eq('Imputar')].tolist()
columnas_indicador = tabla.index[tabla['decisión'].eq('Imputar + indicador')].tolist()
columnas_evaluar = tabla.index[tabla['decisión'].eq('Evaluar')].tolist()

print(f"Imputar: {len(columnas_imputar)}")
print(f"Imputar + indicador: {len(columnas_indicador)}")
print(f"Evaluar: {len(columnas_evaluar)}")
display(pd.Series({'Imputar': len(columnas_imputar),
                   'Imputar + indicador': len(columnas_indicador),
                   'Evaluar': len(columnas_evaluar)}, name='Columnas'))

## 2. Disponibilidad de datos por cohorte

El Issue 4 advierte que parte de los nulos se debe a que ciertas variables comenzaron a registrarse años después. Se calcula el porcentaje de nulos por año de `issue_d`, solo para entender la disponibilidad histórica. No se utiliza la fecha como predictor.

No se recortan años antiguos: eso cambiaría la población de estudio. Si una variable solo está disponible desde 2016, se evalúa conservarla con indicador de ausencia, para que el modelo distinga valores observados de imputados.

In [ ]:
cohortes = entrenamiento[['issue_d']].copy()
cohortes['anio'] = pd.to_datetime(cohortes['issue_d'], errors='coerce').dt.year
cohortes['target'] = np.asarray(y_train)

filas_cohorte = []
for anio, indices in cohortes.groupby('anio').groups.items():
    if pd.isna(anio):
        continue
    fila = {'anio': int(anio), 'n_prestamos': len(indices),
            'Charged Off (%)': cohortes.loc[indices, 'target'].mean() * 100}
    for col in columnas_evaluar:
        fila[f'{col} (% nulos)'] = X_train.loc[indices, col].isna().mean() * 100
    filas_cohorte.append(fila)

tabla_cohortes = pd.DataFrame(filas_cohorte).sort_values('anio')
display(tabla_cohortes.round(1))

## 3. Comparación de las 18 columnas con más de 40 % de nulos

Se comparan tres alternativas en la misma partición de validación:
1. Baseline de nueve variables.
2. Baseline más las 18 columnas, imputadas con mediana.
3. Baseline más las 18 columnas, imputadas con mediana e indicadores de nulo.

La imputación y el escalado se ajustan exclusivamente con entrenamiento. Se reportan ROC-AUC y PR-AUC (Average Precision). PR-AUC se prioriza porque `Charged Off` representa cerca del 20 % de los préstamos.

In [ ]:
BASELINE = ['annual_inc', 'dti', 'open_acc', 'pub_rec', 'revol_bal',
            'total_acc', 'fico_range_low', 'delinq_2yrs', 'inq_last_6mths']

no_numericas = [c for c in columnas_evaluar
                if not pd.api.types.is_numeric_dtype(X_train[c])]
print("Columnas de alta ausencia no numéricas:", no_numericas)
assert not no_numericas, "Revisar el tratamiento de columnas no numéricas."

columnas_ampliadas = BASELINE + [c for c in columnas_evaluar if c not in BASELINE]

def evaluar_configuracion(columnas, indicadores=False):
    modelo = Pipeline([
        ('imputacion', SimpleImputer(strategy='median', add_indicator=indicadores)),
        ('escalado', StandardScaler()),
        ('modelo', LogisticRegression(max_iter=1000, random_state=42))
    ])
    modelo.fit(X_train[columnas], y_train)
    prob = modelo.predict_proba(X_val[columnas])[:, 1]
    return {'ROC-AUC': roc_auc_score(y_val, prob),
            'PR-AUC (AP)': average_precision_score(y_val, prob),
            'variables': len(columnas),
            'indicadores': int(X_train[columnas].isna().any().sum()) if indicadores else 0}

resultados_alta_ausencia = pd.DataFrame({
    'Baseline (9 variables)': evaluar_configuracion(BASELINE),
    'Alta ausencia, mediana': evaluar_configuracion(columnas_ampliadas),
    'Alta ausencia, mediana + indicador': evaluar_configuracion(columnas_ampliadas, True)
}).T
display(resultados_alta_ausencia.round(4))

### Criterio de decisión

Se prioriza PR-AUC y se utiliza ROC-AUC como control complementario. Se conserva el bloque solo si mejora PR-AUC respecto al baseline y ROC-AUC no cae más de 0.002. La comparación es grupal: no demuestra que cada una de las 18 variables aporte valor individual.

In [ ]:
base = resultados_alta_ausencia.loc['Baseline (9 variables)']
opciones = {
    'mediana': resultados_alta_ausencia.loc['Alta ausencia, mediana'],
    'mediana + indicador': resultados_alta_ausencia.loc['Alta ausencia, mediana + indicador']
}
mejor_alternativa = max(opciones, key=lambda k: (opciones[k]['PR-AUC (AP)'],
                                                  opciones[k]['ROC-AUC']))
mejor = opciones[mejor_alternativa]
conservar_bloque_evaluar = (
    mejor['PR-AUC (AP)'] > base['PR-AUC (AP)']
    and mejor['ROC-AUC'] >= base['ROC-AUC'] - 0.002
)
print("Mejor alternativa:", mejor_alternativa)
print("¿Se conserva el bloque?:", conservar_bloque_evaluar)

## 4. Pipeline de imputación

- Las columnas de **Imputar** se completan con mediana (numéricas) o moda (categóricas).
- Las columnas de **Imputar + indicador** se imputan y generan `<columna>_nulo`.
- Las columnas de **Evaluar** se conservan con la alternativa seleccionada en validación, o se eliminan si no superan el baseline.
- Las columnas sin nulos se mantienen.

Los parámetros se aprenden solo de entrenamiento y se aplican sin cambios a validación y prueba.

In [ ]:
class ImputadorPorDecisiones:
    # Aprende estadísticas en fit y las reutiliza en transform.
    def __init__(self, columnas_indicador, columnas_evaluar, conservar_evaluar,
                 indicadores_evaluar):
        self.columnas_indicador = list(columnas_indicador)
        self.columnas_evaluar = list(columnas_evaluar)
        self.conservar_evaluar = conservar_evaluar
        self.indicadores_evaluar = indicadores_evaluar

    def fit(self, X):
        self.columnas_eliminar_ = [] if self.conservar_evaluar else [
            c for c in self.columnas_evaluar if c in X.columns]
        self.parametros_ = {}
        self.indicadores_ = [c for c in self.columnas_indicador if c in X.columns]
        if self.conservar_evaluar and self.indicadores_evaluar:
            self.indicadores_ += [c for c in self.columnas_evaluar
                                  if c in X.columns and X[c].isna().any()]
        for col in X.columns:
            if col in self.columnas_eliminar_ or not X[col].isna().any():
                continue
            if pd.api.types.is_numeric_dtype(X[col]):
                self.parametros_[col] = X[col].median()
            else:
                moda = X[col].mode(dropna=True)
                self.parametros_[col] = moda.iloc[0] if len(moda) else 'Desconocido'
        return self

    def transform(self, X):
        salida = X.drop(columns=self.columnas_eliminar_, errors='ignore').copy()
        for col in self.indicadores_:
            if col in salida.columns:
                salida[f'{col}_nulo'] = salida[col].isna().astype('int8')
        for col, valor in self.parametros_.items():
            if col in salida.columns:
                salida[col] = salida[col].fillna(valor)
        return salida

    def fit_transform(self, X):
        return self.fit(X).transform(X)


imputador = ImputadorPorDecisiones(
    columnas_indicador=columnas_indicador,
    columnas_evaluar=columnas_evaluar,
    conservar_evaluar=conservar_bloque_evaluar,
    indicadores_evaluar=(mejor_alternativa == 'mediana + indicador')
)
imputador.fit(X_train)
X_train_procesado = imputador.transform(X_train)
X_val_procesado = imputador.transform(X_val)
X_test_procesado = imputador.transform(X_test)

print("Estadísticas aprendidas:", len(imputador.parametros_))
print("Indicadores creados:", len(imputador.indicadores_))
print("Columnas eliminadas:", imputador.columnas_eliminar_)

## 5. Validación del resultado

Se verifica que ninguna partición tenga nulos y que todas conserven las mismas columnas y el mismo orden. La prueba solo recibe la transformación ya ajustada; no interviene en el ajuste.

In [ ]:
assert X_train_procesado.isna().sum().sum() == 0
assert X_val_procesado.isna().sum().sum() == 0
assert X_test_procesado.isna().sum().sum() == 0
assert list(X_train_procesado.columns) == list(X_val_procesado.columns)
assert list(X_train_procesado.columns) == list(X_test_procesado.columns)

print("Sin nulos y con columnas consistentes.")
print(f"Train: {X_train_procesado.shape}")
print(f"Validación: {X_val_procesado.shape}")
print(f"Test: {X_test_procesado.shape}")

## 6. Registro de decisiones por columna

La tabla resume la decisión de Issue 4 y el tratamiento final. Se puede copiar al archivo `reports/decisiones_columnas.md`.

In [ ]:
registros = []
for col, fila in tabla.iterrows():
    if fila['decisión'] == 'Imputar':
        final = 'Imputar con mediana/moda'
        motivo = 'Faltantes escasos o diferencia menor al umbral del Issue 4.'
    elif fila['decisión'] == 'Imputar + indicador':
        final = 'Imputar con mediana/moda + indicador'
        motivo = 'El patrón de ausencia se relaciona con la tasa de Charged Off.'
    elif conservar_bloque_evaluar:
        final = ('Conservar e imputar + indicador'
                 if mejor_alternativa == 'mediana + indicador'
                 else 'Conservar e imputar')
        motivo = 'El bloque cumple el criterio de validación frente al baseline.'
    else:
        final = 'Eliminar'
        motivo = 'El bloque no supera al baseline según el criterio definido.'
    registros.append({'columna': col, '% nulos entrenamiento': fila['% nulos'],
                      'MCAR Issue 4': fila['MCAR'],
                      'decisión Issue 4': fila['decisión'],
                      'decisión final Issue 5': final, 'motivo': motivo})

decisiones_issue5 = pd.DataFrame(registros).set_index('columna')
display(decisiones_issue5.round(2))

## 7. Conclusiones

El Issue 5 convierte el diagnóstico de nulos del Issue 4 en un tratamiento reproducible. Las columnas con faltantes escasos se imputan con estadísticas de entrenamiento; las columnas cuyo patrón de ausencia aporta información incorporan indicadores binarios.

Las 18 columnas con más de 40 % de nulos se comparan como bloque en validación, con y sin indicadores. La decisión final depende de las métricas obtenidas. Los nulos asociados a años en los que una variable todavía no se registraba se mantienen diferenciados mediante indicadores, en lugar de recortar toda la población histórica.

**Limitación:** la comparación del bloque no identifica el aporte individual de cada variable. Esa selección puede profundizarse durante el modelado.

## Salida para el Issue 6

Se reemplazan las variables compartidas por las particiones imputadas. El objeto `imputador` conserva las estadísticas aprendidas en entrenamiento y puede reutilizarse para datos nuevos.

In [ ]:
# Única sección que modifica las variables compartidas de la cadena.
X_train = X_train_procesado
X_val = X_val_procesado
X_test = X_test_procesado

assert list(X_train.columns) == list(X_val.columns) == list(X_test.columns)
assert not X_train.isna().any().any()
assert not X_val.isna().any().any()
assert not X_test.isna().any().any()

print(f"X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")
print("Issue 5 completado. Datos listos para Issue 6.")